# Paraphrasing Texts

This notebook paraphrases data from the `Data` folder using the `parrot` model at `github.com/PrithivirajDamodaran/Parrot_Paraphraser.git`.

In addition, we use the `spacy` library for processing texts.

In [1]:
#install spacy if not available
!pip install spacy


[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


In [2]:
import spacy
nlp = spacy.load("en_core_web_sm")

Then read the data. `dataset` can be changed to other data in the `Data` folder. It will also be used to save the paraphrased data.

In [6]:
import pandas as pd

dataset = 'SQUAD_GPT4'
data = pd.read_csv(f'/home/aanis/Metric-Models-for-Detection-of-LLM-Texts/Data/{dataset}.csv')
data.head()

,id,question,answer,gpt1,gpt2
0,0,When did Beyoncé rise to fame?,Beyoncé Giselle Knowles-Carter (/biːˈjɒnseɪ/ b...,Beyoncé rose to fame in the late 1990s as the ...,Beyoncé rose to fame in the late 1990s as the ...
1,6,When did Destiny's Child release their second ...,The group changed their name to Destiny's Chil...,"Destiny's Child, the iconic R&B girl group, re...","Destiny's Child, the iconic American R&B group..."
2,11,Destiny's Child's final album was named what?,"In November 2003, she embarked on the Dangerou...",Destiny's Child's final studio album is titled...,Destiny's Child's final studio album was title...
3,12,How many copies did B'Day sell during the firs...,Beyoncé's second solo album B'Day was released...,"Beyoncé's album ""B'Day"" had a successful first...","Beyoncé's second studio album, ""B'Day,"" had a ..."
4,18,In which year was reports about Beyonce perfor...,"In 2011, documents obtained by WikiLeaks revea...",Reports about Beyoncé performing for Muammar G...,Reports about Beyoncé performing for Muammar G...


Next, we will install the `parrot` library from GitHub to paraphrase the text. We also set up a few functions before starting the paraphrase model. Paraphrasing is done for each sentence in the text

In [7]:
!pip install git+https://github.com/PrithivirajDamodaran/Parrot_Paraphraser.git

  Cloning https://github.com/PrithivirajDamodaran/Parrot_Paraphraser.git to /tmp/pip-req-build-x_xpsnng
  Running command git clone --filter=blob:none --quiet https://github.com/PrithivirajDamodaran/Parrot_Paraphraser.git /tmp/pip-req-build-x_xpsnng
  Resolved https://github.com/PrithivirajDamodaran/Parrot_Paraphraser.git to commit 03084c54b64019ba5fa0b620b9c70ad81123e458
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 134.2/134.2 kB 335.4 kB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 512.4/512.4 kB 1.4 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 1.2 MB/s eta 0:00:0000:0100:010m
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 1.8 MB/s eta 0:00:0000:0100:01
  Created wheel for parrot: filename=parrot-1.0-py3-none-any.whl size=8662 sha256=f1292ba1fe94727d839911d7ab6a0390af927fff55f2724aa7d10

In [9]:
pip install tiktoken

  Using cached tiktoken-0.12.0-cp311-cp311-manylinux_2_28_aarch64.whl.metadata (6.7 kB)
Using cached tiktoken-0.12.0-cp311-cp311-manylinux_2_28_aarch64.whl (1.1 MB)

[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [10]:
from parrot import Parrot
import torch
import warnings
warnings.filterwarnings("ignore")
parrot = Parrot(model_tag="prithivida/parrot_paraphraser_on_T5")

from tqdm import tqdm
tqdm.pandas()

def paraphrase(phrase, a, f):
  doc = nlp(phrase)
  paraphrased = []
  for sent in doc.sents:
    para_phrases = parrot.augment(input_phrase=str(sent),
                                  use_gpu=True,
                                  max_return_phrases = 1,
                                  adequacy_threshold = a,
                                  fluency_threshold = f)
    if para_phrases is None: #if model fails to paraphrase, use original sentence
      paraphrased.append(str(sent))
    else:
      paraphrased.append(para_phrases[0][0])
  return '. '.join(paraphrased)

Could not extract SentencePiece model from /home/aanis/.cache/huggingface/hub/models--prithivida--parrot_paraphraser_on_T5/snapshots/9f32aa1e456e8e8a90d97e8673365f3090fa49fa/spiece.model using sentencepiece library due to 
SentencePieceExtractor requires the protobuf library but it was not found in your environment. Check out the instructions on the
installation page of its repo: https://github.com/protocolbuffers/protobuf/tree/master/python#installation and follow the ones
that match your environment. Please note that you may need to restart your runtime after installation.
. Falling back to TikToken extractor.


ValueError: Error parsing line b'\x0e' in /home/aanis/.cache/huggingface/hub/models--prithivida--parrot_paraphraser_on_T5/snapshots/9f32aa1e456e8e8a90d97e8673365f3090fa49fa/spiece.model

Finally, we run the paraphrase model. This part may take several hours, so please be patient. Also, the code only paraphrases one column at a time. So, you can split among your team to run multiple columns simultaneously. Specifically, one person run `answer`, one run `gpt1`, and one `gpt2`.

In [ ]:
para_answer = data['answer'].progress_apply(lambda x: paraphrase(x,0.5,0.5))
para_gpt1 = data['gpt1'].progress_apply(lambda x: paraphrase(x,0.5,0.5))
para_gpt2 = data['gpt2'].progress_apply(lambda x: paraphrase(x,0.5,0.5))

para_data = pd.DataFrame({
    'answer': para_answer,
    'gpt1': para_gpt1,
    'gpt2': para_gpt2
})

Finally, save the data. Remember to set the correct path and file name to avoid overwriting files (in case you save them in the same place).

In [ ]:
para_data.to_csv('Data/{dataset}_paraphrased.csv')